# 第10回講義 演習

今回は，自己教師あり学習の1つである[Masked Autoencoder](https://openaccess.thecvf.com/content/CVPR2022/papers/He_Masked_Autoencoders_Are_Scalable_Vision_Learners_CVPR_2022_paper.pdf)(MAE)を実装していきます．

## 目次
[【課題】Masked Autoencoderによる自己教師あり学習](#scrollTo=8as0HunPncwp)

1. [Transformer Blockの実装](#scrollTo=S6Os2crtMX5A)

1. [Masked Autoencoderの実装](#scrollTo=J-wkHNcLSog9)

1. [ヘルパー関数の実装](#scrollTo=xiGsQb31qmkm)

1. [学習・推論](#scrollTo=AeXCUI19wPx6)

1. [attention mapの可視化](#scrollTo=s4Mcwi2mLYvl)

1. [Linear probing](#scrollTo=g_DCP3exLYvm)  

　 [参考文献](#scrollTo=atZbN-d1LYvm)



## 【課題】Masked Autoencoderによる自己教師あり学習

MAEではEncoderに入力画像の一部のパッチのみ入力し，DecodeはパッチをEncodeした潜在表現と，学習可能なmask tokenを入力することでもとの画像を再構成するというモデルになっています．MAEの全体像は以下のようになっています．

![MAE overview](https://drive.google.com/uc?id=1pUymdnmbjXCTWmpfXDKgNqb-xi0I0xV2)

MAEのアーキテクチャではTransformerを用いるため，まずTransformer Blockの実装を行い，それらを用いてMAEを実装する流れになっています．



まずは必要なライブラリをimportし，シード値を固定しておきます．  
本演習では`einops`というライブラリを用います．これはテンソルに対する複雑な操作を直感的な記法で実装することができます．演習内では画像をパッチに分割するときや，Attention内でのテンソルの形状の変更，パッチから画像に戻す際に利用します．

document: https://github.com/arogozhnikov/einops

In [ ]:
!pip install einops

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import math
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from einops.layers.torch import Rearrange
from einops import rearrange

In [ ]:
# シード値の設定
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

## 1.Transformer Blockの実装
第8回の演習とほとんど同じです．

**Multi-Head Attention**

TransformerのEncoderはSelf-Attentionになっており，Multi-Headの場合は以下のように表すことができます．

\begin{align}
    Q_{i} &= W^Q_{i} X + b^Q_{i} \\
    K_{i} &= W^K_{i} X + b^K_{i} \\
    V_{i} &= W^V_{i} X + b^V_{i} \\
    head_{i} &= \text{Attention}_i(X) \\
    \text{Multi-Head Attention}(X) &= [head_1, head_2, \cdots , head_n]
\end{align}


In [ ]:
# Multi-Head Attentionの実装
class Attention(nn.Module):
    def __init__(self, dim, heads, dim_head, dropout=0.):
        """
        Arguments
        ---------
        dim : int
            入力データの次元数．埋め込み次元数と一致する
        heads : int
            ヘッドの数
        dim_head : int
            各ヘッドのデータの次元数
        dropout : float
            Dropoutの確率(default=0.)
        """
        super().__init__()

        self.dim = dim
        self.dim_head = dim_head
        inner_dim = dim_head * heads  # ヘッドに分割する前のQ, K, Vの次元数．self.dimと異なっても良い
        project_out = not (heads == 1 and dim_head == dim)  # headsが1，dim_headがdimと等しければ通常のSelf-Attention

        self.heads = heads
        self.scale = math.sqrt(dim_head)  # ソフトマックス関数を適用する前のスケーリング係数(dim_k)

        self.attend = nn.Softmax(dim=-1)  # アテンションスコアの算出に利用するソフトマックス関数
        self.dropout = nn.Dropout(dropout)

        # Q, K, Vに変換するための全結合層
        self.to_q = nn.Linear(in_features=dim, out_features=inner_dim)
        self.to_k = nn.Linear(in_features=dim, out_features=inner_dim)
        self.to_v = nn.Linear(in_features=dim, out_features=inner_dim)

        # dim != inner_dimなら線形層を入れる，そうでなければそのまま出力
        self.to_out = nn.Sequential(
            nn.Linear(in_features=inner_dim, out_features=dim),
            nn.Dropout(dropout),
        ) if project_out else nn.Identity()

    def forward(self, x):
        """
        B: バッチサイズ
        N: 系列長
        D: データの次元数(dim)
        """
        B, N, D = x.size()

        # 入力データをQ, K, Vに変換する
        # (B, N, dim) -> (B, N, inner_dim)
        q = self.to_q(x)
        k = self.to_k(x)
        v = self.to_v(x)

        # Q, K, Vをヘッドに分割する
        # (B, N, inner_dim) -> (B, heads, N, dim_head)
        q = rearrange(q, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)
        k = rearrange(k, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)
        v = rearrange(v, "b n (h d) -> b h n d", h=self.heads, d=self.dim_head)

        # QK^T / sqrt(d_k)を計算する
        # (B, heads, N, dim_head) x (B, heads, dim_head, N) -> (B, heads, N, N)
        dots = torch.matmul(q, k.transpose(-2, -1)) / self.scale

        # ソフトマックス関数でスコアを算出し，Dropoutをする
        attn = self.attend(dots)
        attn = self.dropout(attn)

        # softmax(QK^T / sqrt(d_k))Vを計算する
        # (B, heads, N, N) x (B, heads, N, dim_head) -> (B, heads, N, dim_head)
        out = torch.matmul(attn ,v)

        # もとの形に戻す
        # (B, heads, N, dim_head) -> (B, N, dim)
        out = rearrange(out, "b h n d -> b n (h d)", h=self.heads, d=self.dim_head)

        # 次元が違っていればもとに戻して出力
        # 表現の可視化のためにattention mapも返すようにしておく
        return self.to_out(out), attn

Multi-Head Attentionでは入力の形状と出力の形状が同じになります．実際に確認してみましょう．

今回は入力の次元数と，Attention処理をしたあとの次元数を同じにしています（dim = inner_dim）．

In [ ]:
attn = Attention(384, 12, 32, 0.)
x = torch.rand(4, 64, 384)  # (B, N, D)
attn(x)[0].shape

**Feed-Forward Network**

In [ ]:
# Feed-Forward Networkの実装
class FFN(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        """
        Arguments
        ---------
        dim : int
            入力データの次元数．
        hidden_dim : int
            隠れ層の次元．
        dropout : float
            各全結合層の後のDropoutの確率(default=0.)．
        """
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features=dim, out_features=hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=hidden_dim, out_features=dim),
            nn.Dropout(dropout),
        )  # 隠れ層が1層のMLPで，活性化関数にGELUを使用している

    def forward(self, x):
        """
        (B, D) -> (B, D)
        B: バッチサイズ
        D: 次元数
        """
        return self.net(x)

こちらもAttentionと同様に入力データの次元と出力データの次元は一致しています．実際に確認してみましょう．

In [ ]:
ffn = FFN(384, 768, 0.)
x = torch.rand(4, 64, 384)  # (B, N, D)
ffn(x).shape

**Transformer Block**

上記で実装したAttention, FFNを用いてBlockを実装しましょう．

Blockは以下のような構造をしています．1つのBlockにはAttentionとFFNが1つずつ含まれており，それぞれLayer Normalizationをしてから処理を行い，skip connectionの構造を持っています．

![Block overview](https://drive.google.com/uc?id=1kZc0lCB0FGaht1hQHH8lHHEgAOKViHlE)

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, heads, dim_head, mlp_dim, dropout):
        """
        TransformerのEncoder Blockの実装．

        Arguments
        ---------
        dim : int
            埋め込みされた次元数．PatchEmbedのembed_dimと同じ値．
        heads : int
            Multi-Head Attentionのヘッドの数．
        dim_head : int
            Multi-Head Attentionの各ヘッドの次元数．
        mlp_dim : int
            Feed-Forward Networkの隠れ層の次元数．
        dropout : float
            Droptou層の確率p．
        """
        super().__init__()

        self.attn_ln = nn.LayerNorm(dim)  # Attention前のLayerNorm
        self.attn = Attention(dim, heads, dim_head, dropout)
        self.ffn_ln = nn.LayerNorm(dim)  # FFN前のLayerNorm
        self.ffn = FFN(dim, mlp_dim, dropout)

    def forward(self, x, return_attn=False):
        """
        x: (B, N, dim)
        B: バッチサイズ
        N: 系列長
        dim: 埋め込み次元
        """
        y, attn = self.attn(self.attn_ln(x))
        if return_attn:  # attention mapを返す（attention mapの可視化に利用）
            return attn
        x = y + x
        out = self.ffn(self.ffn_ln(x)) + x

        return out

In [ ]:
block = Block(384, 12, 32, 384, 0.)
x = torch.rand(4, 64, 384)  # (B, N, D)
block(x).shape

**Patch Embedding**

Transformerで画像を扱うために，Patch Embeddingを実装します．


In [ ]:
# Patch Embeddingの実装
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        """
        入力画像をパッチごとに埋め込むための層．

        Arguments
        ---------
        image_size : Tuple[int]
            入力画像のサイズ．
        patch_size : Tuple[int]
            各パッチのサイズ．
        in_channels : int
            入力画像のチャネル数．
        embed_dim : int
            埋め込み後の次元数．
        """
        super().__init__()

        image_height, image_width = image_size
        patch_height, patch_width = patch_size

        assert image_height % patch_height == 0 and image_width % patch_width == 0, "パッチサイズは，入力画像のサイズを割り切れる必要があります．"

        num_patches = (image_height // patch_height) * (image_width // patch_width)  # パッチの数
        patch_dim = in_channels * patch_height * patch_width  # 各パッチを平坦化したときの次元数

        self.to_patch_embedding = nn.Sequential(
            Rearrange("b c (h p1) (w p2) -> b (h w) (p1 p2 c)", p1=patch_height, p2=patch_width),  # 画像をパッチに分割して平坦化
            nn.Linear(in_features=patch_dim, out_features=embed_dim),  # 埋め込みを行う
        )

    def forward(self, x):
        """
        B: バッチサイズ
        C: 入力画像のチャネル数
        H: 入力画像の高さ
        W: 入力画像の幅
        """
        return self.to_patch_embedding(x)  # (B, C, H, W) -> (B, num_patches, embed_dim)

動作確認として画像（CIFAR-10と同じ形状）を模したデータを入力してみましょう．出力として系列データと同様の形状になります．

In [ ]:
patch_emb = PatchEmbedding((32, 32), (4, 4), 3, 384)
x = torch.rand((4, 3, 32, 32))  # (B, C, H, W)
patch_emb(x).shape

## 2.Masked Autoencoderの実装
ここまででMAEを実装するためのコンポーネントを定義することができました．ここからはこれらを用いて実装にMAEを実装していきます．

MAEはAutoencoderと同様の構造をしているため，大きく2つの部分に分けることができます．
- Encoder
- Decoder

Encoderでは入力画像の一部のパッチをランダムに入力して処理します．

DecoderではEncoderからの出力と，mask tokenという特殊なトークンを用いて画像を再構成します．このときにランダムに選択したパッチをもとの順番に戻して入力する必要があります．またEncoderに入力されなかったパッチに対応する部分については，エンコードされた表現が存在しないため代わりにmask tokenを利用します．

Encoder，Decoderを実装する前に，パッチに分割したデータをランダムに選択する関数を先に実装します（`random_indexes`, `take_indexes`）．

`random_indexes`は入力されたパッチをランダムに並べ替えるためのindexを作成する関数です．`forward_indexes`を用いることで入力パッチをランダムに並べ替えることができ，`backward_indexes`を用いることで並べ替えた後のパッチをもとの順番に戻すことができます．

In [ ]:
def random_indexes(size):
    """
    パッチをランダムに並べ替えるためのindexを生成する関数．

    Argument
    --------
    size : int
        入力されるパッチの数（系列長Nと同じ値）．
    """
    forward_indexes = np.arange(size)  # 0からsizeまでを並べた配列を作成
    np.random.shuffle(forward_indexes)  # 生成した配列をシャッフルすることで，パッチの順番をランダムに決定
    backward_indexes = np.argsort(forward_indexes)  # 並べ替えたパッチをもとの順番に戻すためのidx

    return forward_indexes, backward_indexes

実際にどのような挙動をするか確認してみましょう．パッチの代わりに適当な配列を用いて，`random_indexes`で並べ替える，もとに戻すためのindexを作成し実際に並べ替えてみます．

In [ ]:
x = np.random.randn((4))
forward_indexes, backward_indexes = random_indexes(x.shape[0])

print(f"original array: \n{x}")
print(f"forward indexes: \n{forward_indexes}\n{x[forward_indexes]}")  # ランダムに並べ替える
print(f"backward indexes: \n{backward_indexes}\n{x[forward_indexes][backward_indexes]}")  # 並べ替えた後にもとに戻す

このまま利用すると，パッチに分割した形状(B, N, dim)ではパッチごとに並べ替えることができません．実装するためには軸を指定して並べ替える必要が有ります（今回の場合は系列長であるNを指定）．またバッチ内のデータごとに異なる順番にするような実装をします．これを1つの関数として`take_indexes`にまとめて実装します．

In [ ]:
def take_indexes(sequences, indexes):
    """
    パッチを並べ替えるための関数．

    Argument
    --------
    sequences : torch.Tensor
        入力画像をパッチ分割したデータ．(B, N, dim)の形状をしている．
    indexes : np.ndarray
        並べ替えるために利用するindex．
        random_indexesで生成したforward_indexesかbackward_indexesが入ることが想定されている．
    """
    return torch.gather(sequences, dim=1, index=indexes.unsqueeze(2).repeat(1, 1, sequences.shape[-1]))

これらをまとめてパッチ分割したデータを入力して，Encoderに入力するパッチ，並べ替えたときに利用したindex，もとに戻すためのindexを返すようなクラスを実装します．

In [ ]:
class PatchShuffle(nn.Module):
    def __init__(self, ratio):
        # ratio: Encoderに入力しないパッチの割合
        super().__init__()
        self.ratio = ratio

    def forward(self, patches):
        """
        B: バッチサイズ
        N: 系列長（＝パッチの数）
        dim: 次元数（＝埋め込みの次元数）
        """
        B, N, dim = patches.shape
        remain_N = int(N * (1 - self.ratio))  # Encoderに入力するパッチの数

        indexes = [random_indexes(N) for _ in range(B)]  # バッチごとに異なる順番のindexを作る
        forward_indexes = torch.as_tensor(np.stack([i[0] for i in indexes], axis=-1), dtype=torch.long).T.to(patches.device)  # バッチを並べ替えるときのidx (B, N)
        backward_indexes = torch.as_tensor(np.stack([i[1] for i in indexes], axis=-1), dtype=torch.long).T.to(patches.device)  # 並べ替えたパッチをもとの順番に戻すためのidx  (B, N)

        patches = take_indexes(patches, forward_indexes)  # パッチを並べ替える
        patches = patches[:, :remain_N, :]  # Encoderに入力するパッチを抽出

        return patches, forward_indexes, backward_indexes

`PatchShuffle`を用いることで，パッチを並べ替え入力するパッチのみを抽出することと，並べ替えるためのidx，もとの順番に戻すためのidxを得ることができます．

以下の例では，パッチの数がもとのxでは4つありますが，入力するパッチの割合を0.25としているため，処理後の`in_patches`ではパッチの数が各バッチで1つになっています．

In [ ]:
x = torch.rand((4, 4, 8))  # (B, N, dim) 見やすいように小さい行列にしている
patch_shuffle = PatchShuffle(ratio=0.75)
in_patches, forward_idx, backward_idx = patch_shuffle(x)

print(f"original data: shape {x.shape} \n {x}")
print(f"encoder input data: shape {in_patches.shape} \n {in_patches}")

ここからEncoderの実装を行っていきます．Encoderでの処理をまとめると以下のようになります．

1. 入力画像をパッチに分割して（`PatchEmbedding`），positional embeddingする．
2. 分割したパッチをランダムに並べ替えて，必要なパッチのみにする（`PatchShuffle`）．
3. パッチをEncoderに入力して表現を獲得する．

In [ ]:
class MAE_Encoder(torch.nn.Module):
    def __init__(self, image_size=[32, 32], patch_size=[2, 2], emb_dim=192, num_layer=12,
                 heads=3, dim_head=64, mlp_dim=192, mask_ratio=0.75, dropout=0.):
        """
        Arguments
        ---------

        image_size : List[int]
            入力画像の大きさ．
        patch_size : List[int]
            各パッチの大きさ．
        emb_dim : int
            データを埋め込む次元の数．
        num_layer : int
            Encoderに含まれるBlockの数．
        heads : int
            Multi-Head Attentionのヘッドの数．
        dim_head : int
            Multi-Head Attentionの各ヘッドの次元数．
        mlp_dim : int
            Feed-Forward Networkの隠れ層の次元数．
        mask_ratio : float
            入力パッチのマスクする割合．
        dropout : float
            ドロップアウトの確率．
        """
        super().__init__()
        img_height, img_width = image_size
        patch_height, patch_width = patch_size
        num_patches = (img_height // patch_height) * (img_width // patch_width)

        self.cls_token = torch.nn.Parameter(torch.randn(1, 1, emb_dim))  # class tokenの初期化
        self.pos_embedding = torch.nn.Parameter(torch.randn(1, num_patches, emb_dim))  # positional embedding（学習可能にしている）
        self.shuffle = PatchShuffle(mask_ratio)

        # 入力画像をパッチに分割する
        self.patchify = PatchEmbedding(image_size, patch_size, 3, emb_dim)

        # Encoder（Blockを重ねる）
        self.transformer = torch.nn.Sequential(*[Block(emb_dim, heads, dim_head, mlp_dim, dropout) for _ in range(num_layer)])

        self.layer_norm = nn.LayerNorm(emb_dim)

        self.init_weight()

    def init_weight(self):
        torch.nn.init.normal_(self.cls_token, std=0.02)
        torch.nn.init.normal_(self.pos_embedding, std=0.02)

    def forward(self, img):
        # 1. 入力画像をパッチに分割して，positional embeddingする
        patches = self.patchify(img)
        patches =  # WRITE ME

        # 2. 分割したパッチをランダムに並べ替えて，必要なパッチのみ得る
        patches, forward_indexes, backward_indexes =  # WRITE ME ヒント: self.shuffleでパッチのシャッフルと抽出ができる

        # class tokenを結合
        patches = torch.cat([self.cls_token.repeat(patches.shape[0], 1, 1), patches], dim=1)

        # 3. Encoderで入力データを処理する
        features = self.layer_norm(self.transformer(patches))

        return features, backward_indexes

In [ ]:
encoder = MAE_Encoder()

x = torch.rand((4, 3, 32, 32))  # (B, C, H, W)
features, backward_indexes = encoder(x)

print(features.shape)

次にDecoderを実装していきます．Decoderでは以下のような処理を行います．

1. Encoderの入力にmask tokenを結合してからもとの順番に並べ替え，positional embeddingする．
2. Decoderで得られた表現から元画像を再構成する．

In [ ]:
class MAE_Decoder(nn.Module):
    def __init__(self, image_size=[32, 32], patch_size=[2, 2], emb_dim=192, num_layer=4,
                 heads=3, dim_head=64, mlp_dim=192, dropout=0.):
        """
        Arguments
        ---------

        image_size : List[int]
            入力画像の大きさ．
        patch_size : List[int]
            各パッチの大きさ．
        emb_dim : int
            データを埋め込む次元の数．
        num_layer : int
            Decoderに含まれるBlockの数．
        heads : int
            Multi-Head Attentionのヘッドの数．
        dim_head : int
            Multi-Head Attentionの各ヘッドの次元数．
        mlp_dim : int
            Feed-Forward Networkの隠れ層の次元数．
        dropout : float
            ドロップアウトの確率．
        """
        super().__init__()
        img_height, img_width = image_size
        patch_height, patch_width = patch_size
        num_patches = (img_height // patch_height) * (img_width // patch_width)

        self.mask_token = torch.nn.Parameter(torch.rand(1, 1, emb_dim))
        self.pos_embedding = torch.nn.Parameter(torch.rand(1, num_patches+1, emb_dim))

        # Decoder(Blockを重ねる）
        self.transformer = torch.nn.Sequential(*[Block(emb_dim, heads, dim_head, mlp_dim, dropout) for _ in range(num_layer)])

        # 埋め込みされた表現から画像を復元するためのhead
        self.head = torch.nn.Linear(emb_dim, 3 * patch_height * patch_width)
        # (B, N, dim)から(B, C, H, W)にreshapeするためのインスタンス
        self.patch2img = Rearrange("b (h w) (c p1 p2) -> b c (h p1) (w p2)", p1=patch_height, p2=patch_width, h=img_height // patch_height)

        self.init_weight()

    def init_weight(self):
        torch.nn.init.normal_(self.mask_token, std=0.02)
        torch.nn.init.normal_(self.pos_embedding, std=0.02)

    def forward(self, features, backward_indexes):
        # 系列長
        T = features.shape[1]

        # class tokenがある分backward_indexesの最初に0を追加する
        # .toはデバイスの変更でよく利用するが，tensorを渡すことでdtypeを変えることができる
        backward_indexes = torch.cat([torch.zeros(backward_indexes.shape[0], 1).to(backward_indexes), backward_indexes+1], dim=1)

        # 1. mask_tokenを結合して並べ替える．
        # (B, N*(1-mask_ratio)+1, dim) -> (B, N+1, dim)
        features = torch.cat([features, self.mask_token.repeat(features.shape[0], backward_indexes.shape[1] - features.shape[1], 1)], dim=1)
        features =  # WRITE ME ヒント: backward_indexesを用いて元のパッチの順番に戻す
        features =  # WRITE ME

        features = self.transformer(features)

        # class tokenを除去する
        # (B, N+1, dim) -> (B, N, dim)
        features = features[:, 1:, :]

        # 2. 画像を再構成する．
        # (B, N, dim) -> (B, N, 3 * patch_height * patch_width)
        patches = self.head(features)

        # MAEではマスクした部分でのみ損失関数を計算するため，maskも一緒に返す
        mask = torch.zeros_like(patches)
        mask[:, T-1:] = 1  # cls tokenを含めていた分ずらしている
        mask = take_indexes(mask, backward_indexes[:, 1:] - 1)

        img = self.patch2img(patches)
        mask = self.patch2img(mask)

        return img, mask

In [ ]:
decoder = MAE_Decoder()
img, mask = decoder(features, backward_indexes)

print(img.shape)
print(mask.shape)

最後にEncoderとDecoderを合わせてMAE_ViTとして実装しましょう．

In [ ]:
class MAE_ViT(torch.nn.Module):
    def __init__(self, image_size=[32, 32], patch_size=[2, 2], emb_dim=192,
                 enc_layers=12, enc_heads=3, enc_dim_head=64, enc_mlp_dim=768,
                 dec_layers=4, dec_heads=3, dec_dim_head=64, dec_mlp_dim=768,
                 mask_ratio=0.75, dropout=0.):
        """
        Arguments
        ---------
        image_size : List[int]
            入力画像の大きさ．
        patch_size : List[int]
            各パッチの大きさ．
        emb_dim : int
            データを埋め込む次元の数．
        {enc/dec}_layers : int
            Encoder / Decoderに含まれるBlockの数．
        {enc/dec}_heads : int
            Encoder / DecoderのMulti-Head Attentionのヘッドの数．
        {enc/dec}_dim_head : int
            Encoder / DecoderのMulti-Head Attentionの各ヘッドの次元数．
        {enc/dec}_mlp_dim : int
            Encoder / DecoderのFeed-Forward Networkの隠れ層の次元数．
        mask_ratio : float
            入力パッチのマスクする割合．
        dropout : float
            ドロップアウトの確率．
        """
        super().__init__()

        self.encoder = MAE_Encoder(image_size, patch_size, emb_dim, enc_layers,
                                   enc_heads, enc_dim_head, enc_mlp_dim, mask_ratio, dropout)
        self.decoder = MAE_Decoder(image_size, patch_size, emb_dim, dec_layers,
                                   dec_heads, dec_dim_head, dec_mlp_dim, dropout)

    def forward(self, img):
        features, backward_indexes = self.encoder(img)
        rec_img, mask = self.decoder(features, backward_indexes)
        return rec_img, mask

    def get_last_selfattention(self, x):
        patches = self.encoder.patchify(x)
        patches = patches + self.encoder.pos_embedding

        patches = torch.cat([self.encoder.cls_token.repeat(patches.shape[0], 1, 1), patches], dim=1)  # class tokenを結合
        for i, block in enumerate(self.encoder.transformer):
            if i < len(self.encoder.transformer) - 1:
                patches = block(patches)
            else:
                return block(patches, return_attn=True)

In [ ]:
mae = MAE_ViT()
x = torch.rand((4, 3, 32, 32))  # (B, C, H, W)
rec_img, mask = mae(x)

print(rec_img.shape)
print(mask.shape)

## 3.ヘルパー関数の実装
学習に必要なその他の関数を実装していきます．

MAEの学習では学習率のスケジューラにcosine schedulerの使用と，warmupを用います．これらを実装していきます．

In [ ]:
# cosine scheduler
class CosineScheduler:
    def __init__(self, epochs, lr, warmup_length=5):
        """
        Arguments
        ---------
        epochs : int
            学習のエポック数．
        lr : float
            学習率．
        warmup_length : int
            warmupを適用するエポック数．
        """
        self.epochs = epochs
        self.lr = lr
        self.warmup = warmup_length

    def __call__(self, epoch):
        """
        Arguments
        ---------
        epoch : int
            現在のエポック数．
        """
        progress = (epoch - self.warmup) / (self.epochs - self.warmup)
        progress = np.clip(progress, 0.0, 1.0)
        lr = self.lr * 0.5 * (1. + np.cos(np.pi * progress))

        if self.warmup:
            lr = lr * min(1., (epoch+1) / self.warmup)

        return lr

cosine schedulerを用いることで学習率を徐々に小さくすることができます．また，warmupによって最初は徐々に学習率を大きくするという処理になります．

In [ ]:
import matplotlib.pyplot as plt

scheduler = CosineScheduler(epochs=100, lr=0.01)
x = np.arange(100)
plt.plot(x, [scheduler(epoch) for epoch in x])
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.grid(True)

plt.show()

また，optimizerの学習率をepochごとに更新する必要が有ります．そこでoptimizerの学習率を変更できる関数を実装しておきましょう．

In [ ]:
def set_lr(lr, optimizer):
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

## 4.学習・推論

モデルの実装やその他必要な関数を実装したので，実際にモデルを学習してみましょう．まずはハイパーパラメータの設定やモデルの定義を行っていきます．

In [ ]:
# ハイパーパラメータの設定
config = {
    "image_size": [32, 32],
    "patch_size": [2, 2],
    "emb_dim": 192,
    "enc_layers": 12,
    "enc_heads": 3,
    "enc_dim_head": 64,
    "enc_mlp_dim": 192,
    "dec_layers": 4,
    "dec_heads": 3,
    "dec_dim_head": 64,
    "dec_mlp_dim": 192,
    "mask_ratio": 0.75,
    "dropout": 0.
}

In [ ]:
# モデルの定義
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MAE_ViT(**config).to(device)

epochs = 2000
lr = 0.0024
warmup_length = 200
batch_size = 512
step_count = 0
optimizer = optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.05)
scheduler = CosineScheduler(epochs, lr, warmup_length)

次にデータを用意しましょう．今回はCIFAR-10を利用します．

In [ ]:
train_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(0.5, 0.5)]
)
valid_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(0.5, 0.5)]
)

train_dl = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        "./",
        train=True,
        download=True,
        transform=train_transform,
    ),
    batch_size=batch_size,
    shuffle=True
)
valid_dl = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        "./",
        train=False,
        download=True,
        transform=valid_transform,
    ),
    batch_size=batch_size,
    shuffle=False
)

ここからモデルの学習を行っていきます．学習自体は基本的なものと同様ですが，損失関数の計算部分に注意が必要です．

画像の再構成では画像全体で損失関数を計算することも可能ですが，MAEではマスクをかけた部分の損失関数のみを計算しモデルを更新します．そのため損失関数を計算する前にモデルから出力されるmaskをもとの画像を再構成された画像にかける必要があります．

今回はdriveに学習済みモデルがあるので，実際に学習を行わなくても大丈夫です．

In [ ]:
for epoch in range(epochs):
    # スケジューラで学習率を更新する
    new_lr = scheduler(epoch)
    set_lr(new_lr, optimizer)

    total_train_loss = 0.
    total_valid_loss = 0.

    # モデルの訓練
    for x, _ in train_dl:
        step_count += 1
        model.train()
        x = x.to(device)

        rec_img, mask = model(x)
        train_loss = torch.mean((rec_img - x) ** 2 * mask) / config["mask_ratio"]
        train_loss.backward()

        if step_count % 8 == 0:  # 8イテレーションごとに更新することで，擬似的にバッチサイズを大きくしている
            optimizer.step()
            optimizer.zero_grad()

        total_train_loss += train_loss.item()

    # モデルの評価
    with torch.no_grad():
        for x, _ in valid_dl:
            model.eval()

            with torch.no_grad():
                x = x.to(device)

                rec_img, mask = model(x)
                valid_loss = torch.mean((rec_img - x) ** 2 * mask) / config["mask_ratio"]

                total_valid_loss += valid_loss.item()


    print(f"Epoch[{epoch+1} / {epochs}] Train Loss: {total_train_loss/len(train_dl):.4f} Valid Loss: {total_valid_loss/len(valid_dl):.4f}")

# モデルを保存しておく
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/MAE_pretrain_params.pth")

学習が終わったら実際に再構成される画像を見てみましょう．まず保存したモデルを読み込み，`valid_dl`から1バッチ分データを取得し推論します．

In [ ]:
model = MAE_ViT(**config).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/MAE_pretrain_params.pth", map_location=device))

model.eval()
x, _ = next(iter(valid_dl))
with torch.no_grad():
    rec_img, mask = model(x.to(device))

x, rec_img, mask = x.to("cpu"), rec_img.to("cpu"), mask.to("cpu")

次に推論結果を描画してみます．もとのデータを平均0.5，標準偏差0.5で標準化している影響で画素値が0-1ではなくなっているため，まずこれをもとに戻してから描画します．

In [ ]:
fig = plt.figure(figsize=(9, 15))
fig.subplots_adjust(left=0, right=1, bottom=0, top=0.5, hspace=0.05,
                    wspace=0.05)

# MAEの出力そのままを可視化する場合
# imgs = rec_img

# マスクしていた部分は元の画像を用いる
imgs = rec_img * mask + x * (1 - mask)
imgs = (imgs.data + 1) / 2  # 標準化した状態から0-1に戻す
i = 0
for img in imgs[:64]:
    # 出力が線形変換のため0-1になっているとは限らないためclipする
    img = np.clip(np.transpose(torch.squeeze(img).numpy(), (1, 2, 0)), 0, 1)
    ax = fig.add_subplot(8, 8, i+1, xticks=[], yticks=[])
    ax.imshow(img)
    i += 1

以下に実際の画像も描画しています．比較してみると，一部ぼやけていたり色合いなどが異なる部分もありますがある程度画像を復元できていることがわかると思います．

In [ ]:
fig = plt.figure(figsize=(9, 15))
fig.subplots_adjust(left=0, right=1, bottom=0, top=0.5, hspace=0.05,
                    wspace=0.05)

i = 0
imgs = (x.data + 1) / 2
for img in imgs[:64]:
    img = np.clip(np.transpose(torch.squeeze(img).numpy(), (1, 2, 0)), 0, 1)
    ax = fig.add_subplot(8, 8, i+1, xticks=[], yticks=[])
    ax.imshow(img)
    i += 1

## 5.attention mapの可視化
次に表現を獲得するときに，モデルが画像のどの部分に注目しているのかattention mapを可視化して確認してみます．  
今回はEncoderの一番最後のattention mapを可視化してみます．attention mapはheadごとに計算を行っているため，それぞれを見てみましょう．

In [ ]:
def display_attn_map(model, x):
    # Encoderの最後のattention mapを取得
    attn = model.get_last_selfattention(x[0].unsqueeze(0).to(device))

    # Nはパッチの数
    # (1, num_head, N+1, N+1) -> (num_head, N)
    num_head = config["enc_heads"]
    attn = attn[0, :, 0, 1:].reshape(num_head, -1)  # cls tokenに対するスコアを抽出

    val, idx = torch.sort(attn)  # スコアを昇順でソート
    val /= torch.sum(val, dim=1, keepdim=True)  # スコアを[0-1]で正規化する

    # 累積和をとりスコアの合計が0.6ほどになるように残す
    cumval = torch.cumsum(val, dim=1)
    attn = cumval > (1 - 0.4)
    backward_indexes = torch.argsort(idx)

    # ソートしたものを戻す
    for head in range(num_head):
        attn[head] = attn[head][backward_indexes[head]]

    # スコアを画像の形にする
    w_featmap, h_featmap = config["image_size"][0] // config["patch_size"][0], config["image_size"][1] // config["patch_size"][1]
    attn = attn.reshape(num_head, h_featmap, w_featmap).float()

    # 入力画像と同じ大きさにする
    attn = nn.functional.interpolate(attn.unsqueeze(0), scale_factor=config["patch_size"][0], mode="nearest")[0].detach().cpu().numpy()

    # 入力画像とヘッドごとのattention mapを出力する
    fig = plt.figure(figsize=(6, 10))
    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.5, hspace=0.05,
                        wspace=0.05)

    img = (x[0].data + 1) / 2
    img = np.clip(np.transpose(torch.squeeze(img).numpy(), (1, 2, 0)), 0, 1)
    ax = fig.add_subplot(2, 3, 1, xticks=[], yticks=[])
    ax.imshow(img)

    for i in range(len(attn)):
        featmap = attn[i]
        featmap = np.concatenate((featmap[:,:,np.newaxis], np.zeros((32, 32, 2))), axis=2)
        ax = fig.add_subplot(2, 3, i+4, xticks=[], yticks=[])
        ax.imshow(img)
        ax.imshow(featmap, alpha=0.5)


model = MAE_ViT(**config).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/MAE_pretrain_params.pth", map_location=device))
model.eval()
x, _ = next(iter(valid_dl))  # 検証用データからデータを取得する

display_attn_map(model, x)

教師あり学習を用いた場合はどのようなattention mapが得られるのでしょうか？  
事前に教師あり学習でモデルを学習（第8回演習参照）したときのパラメータを準備しているので，こちらを用いて同様にattention mapを可視化してみましょう．  

MAEとは異なりクラス分類をするためのアーキテクチャになるので別にモデルを定義しておきます．

In [ ]:
class ViT_Classifier(nn.Module):
    def __init__(self, encoder: MAE_Encoder, num_classes=10):
        super().__init__()
        self.cls_token = encoder.cls_token
        self.pos_embedding = encoder.pos_embedding
        self.patchify = encoder.patchify
        self.transformer = encoder.transformer
        self.layer_norm = encoder.layer_norm
        self.head = nn.Linear(self.pos_embedding.shape[-1], num_classes)

    def forward(self, img):
        with torch.no_grad():  # エンコーダ部分は勾配計算をしない
          patches = self.patchify(img)
          patches = patches + self.pos_embedding  # positional embedding

          patches = torch.cat([self.cls_token.repeat(patches.shape[0], 1, 1), patches], dim=1)  # class tokenを結合
          features = self.layer_norm(self.transformer(patches))
        logits = self.head(features[:, 0])  # cls tokenのみを入力する
        return logits

    def get_last_selfattention(self, x):
        patches = self.patchify(x)
        patches = patches + self.pos_embedding

        patches = torch.cat([self.cls_token.repeat(patches.shape[0], 1, 1), patches], dim=1)  # class tokenを結合
        for i, block in enumerate(self.transformer):
            if i < len(self.transformer) - 1:
                patches = block(patches)
            else:
                return block(patches, return_attn=True)

In [ ]:
mae = MAE_ViT(**config).to(device)
encoder = mae.encoder

# モデルの定義
model = ViT_Classifier(encoder).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/ViT_supervised_params.pth", map_location=device))
model.eval()

display_attn_map(model, x)

教師あり学習の場合はMAEのattention mapとは異なり，どのheadでも画像の様々な箇所のスコアが大きくなっていることがわかります．

また，今回はそこまできれいに出てはいませんが，他の自己教師あり学習の1つである[DINO](https://arxiv.org/pdf/2104.14294.pdf)では差が明確に出ており，DINOで学習した場合は物体が写っているパッチに対してスコアが大きい部分が集中します．  

![DINO result](https://drive.google.com/uc?id=1gYbTFmW2yLB62lBCQxm-sb0CVzEJmNCk)

## 6.Linear probing
ここまでやってきたことは，MAEの事前学習に当たります．  
最後に事前学習したMAEのEncoderが獲得した表現がどれだけ良いのか評価をしましょう．

今回はLinear probingによって評価します．Linear probingはEncoderの次に分類を行うための線形層を加え，線形層のみを学習する方法です．

実際にLinear probingで評価してみましょう．まずはハイパーパラメータなどの設定と，データローダを定義します．モデルはattention mapの可視化時に利用したものを用います．  

Linear probingでは出力層のみを学習するため，optimizerには出力層のパラメータのみを渡すことに注意してください．  
また損失関数は，事前学習では再構成誤差を用いていましたが，Linear probingではクラス分類になるためcross entropy lossを利用します．  

In [ ]:
# 事前学習をした続きでコードを動かすとメモリが不足します．
# そのため事前学習した後には一度メモリを開放してからLinear probingを動かしてください．
del model
torch.cuda.empty_cache()

In [ ]:
# ハイパーパラメータの設定
config = {
    "image_size": [32, 32],
    "patch_size": [2, 2],
    "emb_dim": 192,
    "enc_layers": 12,
    "enc_heads": 3,
    "enc_dim_head": 64,
    "enc_mlp_dim": 192,
    "dec_layers": 4,
    "dec_heads": 3,
    "dec_dim_head": 64,
    "dec_mlp_dim": 192,
    "mask_ratio": 0.75,
    "dropout": 0.
}

device = "cuda" if torch.cuda.is_available() else "cpu"
pretrained_model = MAE_ViT(**config).to(device)
pretrained_model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/MAE_pretrain_params.pth", map_location=device))

encoder = pretrained_model.encoder

# モデルの定義
model = ViT_Classifier(encoder).to(device)

epochs = 100
lr = 0.0005
warmup_length = 5
batch_size = 128
optimizer = optim.AdamW(model.head.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=0.05)  # 分類器部分のみ学習
scheduler = CosineScheduler(epochs, lr, warmup_length)
criterion = nn.CrossEntropyLoss()

In [ ]:
train_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(0.5, 0.5)]
)
valid_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(0.5, 0.5)]
)

train_dl = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        "./",
        train=True,
        download=True,
        transform=train_transform,
    ),
    batch_size=batch_size,
    shuffle=True
)
valid_dl = torch.utils.data.DataLoader(
    datasets.CIFAR10(
        "./",
        train=False,
        download=True,
        transform=valid_transform,
    ),
    batch_size=batch_size,
    shuffle=False
)

準備ができたので分類器を学習させてみましょう．事前学習では明示的に分類できるような表現を学習しているわけではないですが，得られた表現を用いることでクラス分類ができていることがわかります．

In [ ]:
for epoch in range(epochs):
    new_lr = scheduler(epoch)
    set_lr(new_lr, optimizer)

    total_train_loss = 0.
    total_train_acc = 0.
    total_valid_loss = 0.
    total_valid_acc = 0.
    for x, t in train_dl:
        x, t = x.to(device), t.to(device)
        pred = model(x)

        train_loss = criterion(pred, t)
        train_acc = (torch.argmax(pred, dim=1) == t).float().mean().cpu()

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        total_train_loss += train_loss.item()
        total_train_acc += train_acc

    with torch.no_grad():
        for x, t in valid_dl:
            x, t = x.to(device), t.to(device)
            pred = model(x)

            valid_loss = criterion(pred, t)
            valid_acc = (torch.argmax(pred, dim=1) == t).float().mean().cpu()

            total_valid_loss += valid_loss.item()
            total_valid_acc += valid_acc

    print(f"Epoch[{epoch+1} / {epochs}]",
          f"Train Loss: {total_train_loss/len(train_dl):.4f}",
          f"Train Acc.: {total_train_acc/len(train_dl):.4f}",
          f"Valid Loss: {total_valid_loss/len(valid_dl):.4f}",
          f"Valid Acc.: {total_valid_acc/len(valid_dl):.4f}")

torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/DLBasics2025_colab/Lecture10/model/MAE_classifier_params.pth")

## 参考文献
[1] [Masked Autoencoders Are Scalable Vision Learners](https://arxiv.org/pdf/2111.06377.pdf)  
[2] [AN IMAGE IS WORTH 16X16 WORDS: TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE](https://arxiv.org/pdf/2010.11929.pdf)  
[3] [Emerging Properties in Self-Supervised Vision Transformers](https://arxiv.org/pdf/2104.14294.pdf)